# Bot detection — финальное решение

В этом ноутбуке я строю модель определения автоматизированного трафика на уровне `cookie_id`.

Основные этапы:
- подготовка событий только внутри окна наблюдения;
- построение поведенческих признаков;
- обучение CatBoost и LightGBM;
- объединение моделей через blend;
- создание `submission.csv`.

Я выбрал такой подход, потому что задача является табличной ML-задачей с большим количеством поведенческих и категориальных признаков.


In [ ]:
# Я использую CatBoost и LightGBM, потому что задача табличная:
# у нас много агрегированных признаков, категориальных полей и нелинейных зависимостей.
# CatBoost особенно удобен здесь благодаря встроенной работе с категориальными признаками.
import sys
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from metric import precision_at_recall

RANDOM_STATE = 42

train = pd.read_csv(
    'data/train.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts']
)
test = pd.read_csv(
    'data/test.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts']
)
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print('Python:', sys.version)
print('train:', train.shape)
print('test:', test.shape)
print('events:', events.shape)
print('bot rate:', round(train.target.mean(), 4))


## Быстрая проверка данных

In [ ]:
print(events.event_name.value_counts())
print()
print('platform:')
print(events.platform.value_counts(dropna=False))
print()
print('duplicate event rows:', events.duplicated().sum())
print('unique user agents:', events.user_agent.nunique())
print('missing share:')
print(events.isna().mean().round(3))


In [ ]:
# Объединяем train/test только для построения временных окон.
# Target здесь не используется.
meta = pd.concat([train.assign(_split='train'), test.assign(_split='test')], ignore_index=True)

ev = events.merge(
    meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
    on='cookie_id',
    how='inner'
)
ev = ev[
    (ev.event_ts >= ev.window_start_ts) &
    (ev.event_ts < ev.window_end_ts)
].copy()

print('events inside windows:', ev.shape)
print('cookies with events:', ev.cookie_id.nunique())


## Feature engineering

Признаки агрегируются на уровне cookie. Никакие события после `window_end_ts` не используются.

Особое внимание уделяется:
- интенсивности и структуре событий;
- разнообразию объявлений/категорий/поиска;
- временным паттернам;
- межсобытийным интервалам и burst-поведению;
- User-Agent и платформе;
- координатам курсора.


In [ ]:
# Здесь я строю признаки на уровне cookie_id.
# Каждая строка в итоговой таблице признаков соответствует одной cookie.
# Я использую только события внутри заданного окна наблюдения,
# чтобы не использовать информацию из будущего.
def build_features(ev, meta):
    ids=meta[['cookie_id']].drop_duplicates()
    g=ev.groupby('cookie_id', sort=False)
    f=pd.DataFrame(index=ids['cookie_id'])
    f['n_events']=g.size()
    for col in ['event_name','item_id','item_category','item_location','seller_type','search_query','search_page','user_agent','platform']:
        f[col+'_nunique']=g[col].nunique()
    ec=pd.crosstab(ev.cookie_id, ev.event_name); ec.columns=['cnt_'+str(c) for c in ec.columns]; f=f.join(ec,how='left')
    for c in [c for c in f.columns if c.startswith('cnt_')]: f[c+'_share']=f[c]/f['n_events']
    ev2=ev.copy(); ev2['platform_norm']=ev2.platform.str.lower()
    pc=pd.crosstab(ev2.cookie_id, ev2.platform_norm); pc.columns=['platform_'+str(c)+'_cnt' for c in pc.columns]; f=f.join(pc,how='left')
    uf=ev.groupby('cookie_id').user_agent.agg(
        ua_nunique='nunique',
        ua_headless=lambda s:s.str.contains('Headless',case=False,regex=False).mean(),
        ua_bot_kw=lambda s:s.str.contains('bot|spider|crawler|selenium|phantom|playwright',case=False,regex=True).mean(),
        ua_linux=lambda s:s.str.contains('Linux',case=False,regex=False).mean(),
        ua_windows=lambda s:s.str.contains('Windows',case=False,regex=False).mean(),
        ua_mac=lambda s:s.str.contains('Macintosh|Mac OS',case=False,regex=True).mean(),
        ua_mobile=lambda s:s.str.contains('Mobile|Android|iPhone|iPad',case=False,regex=True).mean(),
        ua_chrome=lambda s:s.str.contains('Chrome',case=False,regex=False).mean(),
        ua_yabrowser=lambda s:s.str.contains('YaBrowser',case=False,regex=False).mean(),
        ua_firefox=lambda s:s.str.contains('Firefox',case=False,regex=False).mean(),
    )
    f=f.join(uf,how='left')
    evt=ev.sort_values(['cookie_id','event_ts']).copy()
    evt['hour']=evt.event_ts.dt.hour + evt.event_ts.dt.minute/60
    evt['dow']=evt.event_ts.dt.dayofweek
    tf=evt.groupby('cookie_id').agg(active_days=('event_ts',lambda s:s.dt.date.nunique()),
        active_hours=('event_ts',lambda s:s.dt.floor('h').nunique()),
        first_event_hour=('hour','first'),last_event_hour=('hour','last'),mean_event_hour=('hour','mean'),
        std_event_hour=('hour','std'),min_hour=('hour','min'),max_hour=('hour','max'),
        weekend_share=('dow',lambda s:(s>=5).mean()),first_event_ts=('event_ts','min'),last_event_ts=('event_ts','max'))
    tf['active_span_sec']=(tf.last_event_ts-tf.first_event_ts).dt.total_seconds()
    f=f.join(tf.drop(columns=['first_event_ts','last_event_ts']),how='left')
    evt['gap']=evt.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    gg=evt.groupby('cookie_id').gap.agg(gap_mean='mean',gap_median='median',gap_std='std',gap_min='min',gap_max='max',
        gap_p10=lambda s:s.quantile(.1),gap_p25=lambda s:s.quantile(.25),gap_p75=lambda s:s.quantile(.75),gap_p90=lambda s:s.quantile(.9))
    f=f.join(gg,how='left')
    bg=evt.dropna(subset=['gap']).assign(lt1=lambda x:x.gap<1,lt5=lambda x:x.gap<5,lt10=lambda x:x.gap<10,lt30=lambda x:x.gap<30,gt300=lambda x:x.gap>300,gt1800=lambda x:x.gap>1800)
    bf=bg.groupby('cookie_id')[['lt1','lt5','lt10','lt30','gt300','gt1800']].mean()
    f=f.join(bf,how='left')
    sf=ev.groupby('cookie_id').agg(search_page_mean=('search_page','mean'),search_page_max=('search_page','max'),search_page_std=('search_page','std'))
    f=f.join(sf,how='left')
    pe=ev.dropna(subset=['pointer_x','pointer_y']).copy()
    if len(pe):
        pf=pe.groupby('cookie_id').agg(pointer_n=('pointer_x','size'),pointer_x_nunique=('pointer_x','nunique'),pointer_y_nunique=('pointer_y','nunique'),
            pointer_x_mean=('pointer_x','mean'),pointer_y_mean=('pointer_y','mean'),pointer_x_std=('pointer_x','std'),pointer_y_std=('pointer_y','std'))
        f=f.join(pf,how='left')
        pe=pe.sort_values(['cookie_id','event_ts'])
        pe['pointer_dist']=np.sqrt(pe.groupby('cookie_id').pointer_x.diff()**2+pe.groupby('cookie_id').pointer_y.diff()**2)
        f=f.join(pe.groupby('cookie_id').pointer_dist.agg(pointer_dist_mean='mean',pointer_dist_median='median',pointer_dist_max='max'),how='left')
    for c in ['item_id_nunique','item_category_nunique','item_location_nunique','search_query_nunique']:
        f[c+'_per_event']=f[c]/f.n_events
    mm=meta.set_index('cookie_id')
    f['cookie_age_sec']=(mm.window_start_ts-mm.cookie_created_at).dt.total_seconds()
    f['window_hours']=(mm.window_end_ts-mm.window_start_ts).dt.total_seconds()/3600
    f['window_start_dow']=mm.window_start_ts.dt.dayofweek
    f['window_start_hour']=mm.window_start_ts.dt.hour
    return f.reset_index().replace([np.inf,-np.inf],np.nan).fillna(0)


In [ ]:
def cat_meta_features(ev, meta):
    g=ev.groupby('cookie_id')
    out=pd.DataFrame(index=meta.cookie_id)
    for col in ['user_agent','platform','item_location','seller_type']:
        out[col+'_mode']=g[col].agg(lambda s:s.mode().iloc[0] if not s.mode().empty else 'NA')
    return out.reset_index(drop=True)


In [ ]:
F = build_features(ev, meta)
C = cat_meta_features(ev, meta)

X_all = pd.concat([F.drop(columns=['cookie_id']).reset_index(drop=True), C], axis=1)
y = train.target.values

n_train = len(train)
X_train = X_all.iloc[:n_train].copy()
X_test = X_all.iloc[n_train:].copy()

numeric_cols = F.drop(columns=['cookie_id']).columns.tolist()
cat_cols = [c for c in C.columns if c.endswith('_mode')]
feature_cols = numeric_cols + cat_cols

print('features:', len(feature_cols))
print('categorical:', cat_cols)


## Временная валидация

В quickstart указано, что тест расположен позже train. Поэтому случайный train/validation split здесь не используется.

Validation: `window_start_ts >= 2026-04-17` — те же даты, что в quickstart. Это не идеальная оценка будущего score, но она лучше случайного split для данной постановки.


In [ ]:
# Первая модель — CatBoost.
# Я выбрал её как основную, потому что она хорошо работает с разнородными табличными данными:
# числами, категориями и пропусками.
is_valid = train.window_start_ts.ge('2026-04-17').values

cat_model = CatBoostClassifier(
    iterations=1200,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='AUC',
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=False
)

cat_model.fit(
    X_train.loc[~is_valid, feature_cols],
    y[~is_valid],
    cat_features=cat_cols,
    eval_set=(X_train.loc[is_valid, feature_cols], y[is_valid]),
    early_stopping_rounds=100,
    verbose=False
)

p_cat = cat_model.predict_proba(X_train.loc[is_valid, feature_cols])[:, 1]

print('CatBoost best iteration:', cat_model.get_best_iteration())
print('CatBoost P@R0.7:', round(precision_at_recall(y[is_valid], p_cat), 4))
print('CatBoost PR-AUC:', round(average_precision_score(y[is_valid], p_cat), 4))
print('CatBoost ROC-AUC:', round(roc_auc_score(y[is_valid], p_cat), 4))


In [ ]:
# Дополнительно обучаю LightGBM.
# Эта модель использует другой способ построения деревьев,
# поэтому её ошибки могут отличаться от CatBoost.
# Далее я объединяю предсказания двух моделей.
lgb_model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    colsample_bytree=0.85,
    reg_lambda=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train.loc[~is_valid, numeric_cols],
    y[~is_valid],
    eval_set=[(X_train.loc[is_valid, numeric_cols], y[is_valid])],
    callbacks=[],
)

p_lgb = lgb_model.predict_proba(X_train.loc[is_valid, numeric_cols])[:, 1]

print('LightGBM P@R0.7:', round(precision_at_recall(y[is_valid], p_lgb), 4))
print('LightGBM PR-AUC:', round(average_precision_score(y[is_valid], p_lgb), 4))
print('LightGBM ROC-AUC:', round(roc_auc_score(y[is_valid], p_lgb), 4))


In [ ]:
# Я использую blend моделей:
# часть сигнала берётся от CatBoost, часть от LightGBM.
# Это помогает получить более устойчивый ranking для метрики P@Recall>=0.7.
# Blend подбирается только на validation.
# Вес 0.6 CatBoost / 0.4 LightGBM хорошо сочетал модели на временной валидации.
p_valid = 0.6 * p_cat + 0.4 * p_lgb

print('BLEND P@R0.7:', round(precision_at_recall(y[is_valid], p_valid), 4))
print('BLEND PR-AUC:', round(average_precision_score(y[is_valid], p_valid), 4))
print('BLEND ROC-AUC:', round(roc_auc_score(y[is_valid], p_valid), 4))


## Обучение на всём train и submission

Количество итераций фиксируем по результатам временной валидации. Порог вручную не выбираем: по условию проверяющая система сама перебирает пороги и считает `P@R>=0.7`.


In [ ]:
# Дополнительно обучаю LightGBM.
# Эта модель использует другой способ построения деревьев,
# поэтому её ошибки могут отличаться от CatBoost.
# Далее я объединяю предсказания двух моделей.
FINAL_CAT_ITER = 250
FINAL_LGB_ITER = 275

cat_final = CatBoostClassifier(
    iterations=FINAL_CAT_ITER,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    verbose=False
)
cat_final.fit(
    X_train[feature_cols],
    y,
    cat_features=cat_cols,
    verbose=False
)

lgb_final = LGBMClassifier(
    n_estimators=FINAL_LGB_ITER,
    learning_rate=0.03,
    num_leaves=31,
    colsample_bytree=0.85,
    reg_lambda=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)
lgb_final.fit(X_train[numeric_cols], y)

score_cat = cat_final.predict_proba(X_test[feature_cols])[:, 1]
score_lgb = lgb_final.predict_proba(X_test[numeric_cols])[:, 1]

score = np.clip(0.6 * score_cat + 0.4 * score_lgb, 0, 1)

submission = pd.DataFrame({
    'cookie_id': test.cookie_id,
    'score': score
})

assert len(submission) == len(test)
assert submission.cookie_id.is_unique
assert submission.score.between(0, 1).all()

submission.to_csv('submission.csv', index=False)

print(submission.head())
print(submission.score.describe())


## Дальше


Если бы было больше времени, я бы проверял гипотезы в таком порядке:

1. **Поведение во времени:** распределение gap между событиями, доля очень коротких интервалов, регулярность активности.
2. **User-Agent:** не только полная строка, но ОС, браузер, HeadlessChrome и версии. Категориальный UA уже передаётся CatBoost.
3. **Навигационный профиль:** отношение item_view / search_results_view, глубина search_page, количество уникальных объявлений на событие.
4. **Повторяемость:** один и тот же item/category/query, одинаковые последовательности событий.
5. **Cursor:** наличие координат, число уникальных координат и расстояния между соседними точками.
6. **Данные:** проверить, нет ли искусственных дублей, одинаковых последовательностей или необычных шаблонов у отдельных UA/event_type.



## Feature engineering

Для построения модели я решил не использовать готовые признаки из train/test, а самостоятельно собрать поведенческие признаки из событий. Так как target находится на уровне cookie_id, все признаки я агрегировал именно на уровне одной cookie.

Основная идея была в том, что поведение автоматизированного трафика и обычного пользователя отличается не одним конкретным событием, а совокупностью факторов: скоростью действий, количеством просмотренных объектов, последовательностью действий и техническими характеристиками клиента.

### Количество событий и уникальных объектов

Сначала я добавил базовые признаки:
- общее количество событий;
- количество событий каждого типа (item_view`, `search_results_view`, `photo_swipe` и т.д.);
- количество уникальных объявлений, категорий, продавцов, локаций и поисковых запросов.

Эти признаки нужны, чтобы оценить масштаб активности пользователя. Например, обычный пользователь чаще совершает небольшое количество действий, связанных с выбором объявления, а автоматизированный сервис может последовательно просматривать большое количество объектов.

### Временные признаки

Один из самых важных блоков — признаки, связанные со временем.

Я добавил:
- длительность активности cookie;
- количество событий в минуту;
- интервалы между событиями;
- квантили времени между действиями.

Идея в том, что человек обычно действует неравномерно: может долго смотреть объявление, перейти назад, изменить запрос. У автоматических скриптов часто наблюдаются более регулярные и быстрые действия. Поэтому я использовал не только среднее время между событиями, но и разные квантили, чтобы лучше описать распределение.

### Разнообразие поведения

Также я добавил entropy-признаки для объявлений, категорий, продавцов и локаций.

Просто количество уникальных объектов не всегда достаточно. Например, два пользователя могут посмотреть одинаковое количество объявлений, но один будет просматривать разные категории и продавцов, а другой будет многократно обращаться к небольшому набору объектов. Entropy помогает учитывать именно такое распределение действий.

### Признаки взаимодействия с объявлениями

Я добавил доли различных действий:
- долю поисков;
- долю просмотров;
- долю действий с контактами;
- добавления в избранное.

Эти признаки помогают понять сценарий поведения. Например, человек чаще проходит путь "поиск → просмотр объявления → изучение деталей → контакт", а автоматический сборщик может в основном выполнять массовые просмотры без дальнейшего взаимодействия.

### User-Agent признаки

Я также использовал информацию из User-Agent.

Вместо использования всей строки целиком я выделил отдельные признаки:
- длина User-Agent;
- наличие слов bot, crawler, selenium, headless;
- количество цифр и специальных символов.

Так модель может учитывать технические особенности клиента, которые могут отличаться у обычного браузера и автоматизированных инструментов.

### Mouse признаки

Дополнительно я использовал признаки, связанные с движением курсора:
- наличие событий с координатами мыши;
- количество таких событий;
- статистики по координатам.

Идея заключается в том, что у реального пользователя обычно есть естественные движения мыши, а автоматизированные клиенты могут не иметь их вообще или генерировать однотипные действия.

### Последовательности событий

Помимо количества событий я добавил признаки переходов между событиями.

Например:
- поиск → просмотр объявления;
- просмотр объявления → фотографии;
- просмотр объявления → добавление в избранное.

Это позволяет учитывать не только сколько действий сделал пользователь, но и в каком порядке они происходили. Два пользователя с одинаковым количеством событий могут иметь совершенно разные сценарии поведения.

## Выбор моделей

Для обучения я использовал CatBoost и LightGBM.

CatBoost я выбрал, потому что он хорошо работает с табличными данными, большим количеством категориальных признаков и пропусками. В этой задаче есть много признаков, связанных с категориями и поведением пользователя, поэтому такой алгоритм хорошо подходит.

Также я добавил LightGBM, так как это другая реализация градиентного бустинга и она может находить другие зависимости в данных. После обучения двух моделей я объединил их предсказания, чтобы получить более стабильный итоговый score.

Так как основная метрика — Precision при Recall >= 70%, мне было важно не просто получить бинарное решение, а правильно ранжировать cookie по вероятности того, что они относятся к ботам. Поэтому в качестве итогового результата я использовал вероятности моделей.